# v4 vs. the Literature — 5 Standard Tabular Benchmarks (GPU)

Runs v4's two best arms from the last comparison (mean-pool and causal attention, each
with/without the PLR per-feature embeddings) next to a tuned XGBoost and PlainMLP, on
**5 datasets with their exact published train/val/test splits**, so the results are directly
comparable to the FT-Transformer paper's own table.

| Key | Dataset | Rows | Task | Metric |
|---|---|---|---|---|
| CA | California Housing | 20,640 | regression | RMSE ↓ |
| AD | Adult | 48,842 | binary | accuracy ↑ |
| JA | Jannis | 83,733 | multiclass (4) | accuracy ↑ |
| CO | Covertype | 581,012 | multiclass (7) | accuracy ↑ |
| YE | YearPredictionMSD | 515,345 | regression | RMSE ↓ |

Source of splits and published numbers: Gorishniy, Rubachev, Khrulkov & Babenko (2021),
*Revisiting Deep Learning Models for Tabular Data*, NeurIPS 2021 (arXiv:2106.11959) — the
FT-Transformer paper. Published table has NO reported std (unlike the num-embeddings paper
used in notebook 07), so treat the published numbers as single reference points, not
distributions.

### Scale warning
CO and YE are far larger than anything run so far (up to 580k rows). Budget real GPU time
and use the env-var overrides below for a smoke test before committing to the full run.

### Fairness notes
* v4 is **not tuned per dataset** (uses the config from notebook 06). Every published model
  was tuned per dataset. XGBoost here gets its own per-dataset tuning, same budget on every
  dataset, so it isn't a straw man.
* V1 is intentionally **not** included here — it already lost to v4 on 3 datasets in
  notebook 07; re-running it on 2 much larger datasets would roughly double this notebook's
  runtime for a question already answered.

## 0. Setup

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")
PROJECT_DIR = os.getcwd(); sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch, optuna
import matplotlib.pyplot as plt, seaborn as sns
from joblib import Parallel, delayed

from benchmarks_rtdl import BENCHMARKS, load_benchmark, published, higher_is_better
from training import get_device

DEVICE = get_device()

# Knobs, overridable from the shell so a headless run needs no edits:
#   SRP_SUBSAMPLE=2000 SRP_SEEDS=1 SRP_XGB_TRIALS=2 SRP_EPOCHS=2 SRP_KEYS=CA jupyter nbconvert ...
KEYS       = os.environ.get("SRP_KEYS", "CA,AD,JA,CO,YE").split(",")
SEEDS      = tuple(range(int(os.environ.get("SRP_SEEDS", "3"))))
XGB_TRIALS = int(os.environ.get("SRP_XGB_TRIALS", "25"))
SUBSAMPLE  = int(os.environ.get("SRP_SUBSAMPLE", "0")) or None
EPOCHS     = int(os.environ.get("SRP_EPOCHS", "60"))
PATIENCE   = int(os.environ.get("SRP_PATIENCE", "8"))
V4_TRAIN_KW = dict(epochs=EPOCHS, lr=1e-3, batch_size=512, weight_decay=1e-4, patience=PATIENCE)
EMB_KW = dict(d_embedding=8, n_frequencies=16, sigma=0.1)
ENS_KW = dict(num_learners=16, hidden_dim=16, embed_dim=32, depth=1, dropout=0.1, feature_frac=0.7)
ATT_KW = dict(embed_dim=32, num_heads=4, attn_depth=2, ff_mult=4, dropout=0.1, need_weights=False)

MODELS = [
    ("v4 mean-pool",         "v4",  "meanpool_wide", "none"),
    ("v4 mean-pool + PLR",   "v4",  "meanpool_wide", "periodic"),
    ("v4 causal attn",       "v4",  "causal",        "none"),
    ("v4 causal attn + PLR", "v4",  "causal",        "periodic"),
    ("PlainMLP",             "mlp", None,            None),
]
THREADS_PER_JOB = 2
N_JOBS = int(os.environ.get("SRP_JOBS", "0")) or (
    3 if DEVICE.type == "cuda" else max(1, min(8, (os.cpu_count() or 2) // THREADS_PER_JOB)))

RUN_DIR = os.path.join(PROJECT_DIR, "runs"); os.makedirs(RUN_DIR, exist_ok=True)
STAMP = time.strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(RUN_DIR, f"08_progress_{STAMP}.log")
_latest = os.path.join(RUN_DIR, "08_progress.log")
if os.path.islink(_latest) or os.path.exists(_latest):
    os.remove(_latest)
os.symlink(os.path.basename(LOG_PATH), _latest)
def log(msg):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f: f.write(time.strftime("%H:%M:%S ") + msg + "\n")
def show(df, fmt=None, style=None):
    try:
        s = df.style.format(fmt or {}, na_rep="—"); display(style(s) if style else s)
    except (AttributeError, ImportError):
        display(df.round(4))

sns.set_theme(style="whitegrid", context="notebook")
optuna.logging.set_verbosity(optuna.logging.WARNING)
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only"
log(f"device: {DEVICE} ({gpu}) | torch {torch.__version__} | {N_JOBS} parallel jobs | "
    f"keys={KEYS} seeds={len(SEEDS)} epochs={EPOCHS}")
if DEVICE.type != "cuda":
    print("NOTE: running on CPU — CO and YE will be very slow here. Meant for the GPU server.")

## 1. Load the 5 benchmarks (paper-exact splits, verified)

In [ ]:
DATA = {}
for k in KEYS:
    d = load_benchmark(k)
    if SUBSAMPLE:
        n = SUBSAMPLE
        d.X_tr, d.y_tr = d.X_tr[:n], d.y_tr[:n]
        d.X_va, d.y_va = d.X_va[:n // 2], d.y_va[:n // 2]
        d.X_te, d.y_te = d.X_te[:n], d.y_te[:n]
    DATA[k] = d
    log(d.summary())
METRIC = {k: ("rmse" if DATA[k].task == "regression" else "acc") for k in KEYS}
if SUBSAMPLE:
    print(f"SMOKE TEST on {SUBSAMPLE} rows — numbers below are not comparable to the paper.")

## 2. Score every model the same way

In [ ]:
def score_from_raw(data, y_true, raw):
    # raw = logits for binary/multiclass classification, standardised predictions for regression
    raw = np.asarray(raw).reshape(len(y_true), -1)
    if data.task == "regression":
        return float(np.sqrt(np.mean((raw[:, 0] - y_true) ** 2)) * data.y_std)
    if data.task == "binary":
        return float(((raw[:, 0] > 0).astype(int) == y_true).mean())
    return float((raw.argmax(1) == y_true).mean())

print("published single-model results on these exact splits (for reference; no std reported)")
FOCUS = ["MLP", "ResNet", "NODE", "FT-Transformer"]
show(pd.DataFrame({BENCHMARKS[k]["label"]: {m: published(k, m) for m in FOCUS if True}
                   for k in KEYS}))

## 3. Train everything

5 models x up to 5 datasets x seeds. Slowest (largest) datasets scheduled first so the
worker pool stays busy until the end. This is the long cell.

In [ ]:
def run_one(label, kind, agg, mode, key, seed, arrays, task_kind, output_dim, n_features,
            ens_kw, att_kw, emb_kw, train_kw, device_str, project_dir, threads):
    import sys, time
    if project_dir not in sys.path: sys.path.insert(0, project_dir)
    import numpy as np, torch
    torch.set_num_threads(threads)
    from training import train_model, predict, to_tensors
    t0 = time.time()
    data = to_tensors(arrays, device_str)
    torch.manual_seed(seed); np.random.seed(seed)

    if kind == "mlp":
        from model import PlainMLP
        model = PlainMLP(n_features, task=task_kind, output_dim=output_dim)
    else:
        from weak_learners import WeakLearnerConfig
        from attention import AttentionConfig
        from embeddings import EmbeddingConfig
        from model import make_arm
        model = make_arm(agg, n_features, WeakLearnerConfig(**ens_kw, seed=seed, batched=True),
                         AttentionConfig(**att_kw), task=task_kind, output_dim=output_dim,
                         embedding=EmbeddingConfig(mode=mode, **emb_kw))
    model, hist = train_model(model, data, seed=seed, **train_kw)
    return {"label": label, "key": key, "seed": seed, "raw": predict(model, data["Xte"]),
            "n_params": sum(p.numel() for p in model.parameters()),
            "seconds": time.time() - t0, "epochs": hist["best_epoch"] + 1}

SIZE_ORDER = {"CO": 0, "YE": 1, "AD": 2, "JA": 3, "CA": 4}   # largest first
jobs = [(lbl, kind, agg, mode, k, s) for k in KEYS for (lbl, kind, agg, mode) in MODELS for s in SEEDS]
jobs.sort(key=lambda j: SIZE_ORDER.get(j[4], 9))
log(f"{len(jobs)} runs on {N_JOBS} workers ({DEVICE}) ...")

t0, RES = time.time(), []
for (lbl, kind, agg, mode, k, s), r in zip(jobs, Parallel(n_jobs=N_JOBS, return_as="generator")(
        delayed(run_one)(lbl, kind, agg, mode, k, s, DATA[k].arrays(), DATA[k].task,
                         DATA[k].output_dim, DATA[k].n_features, ENS_KW, ATT_KW, EMB_KW,
                         V4_TRAIN_KW, str(DEVICE), PROJECT_DIR, THREADS_PER_JOB)
        for lbl, kind, agg, mode, k, s in jobs)):
    d = DATA[k]
    r["score"] = score_from_raw(d, d.y_te, r.pop("raw"))
    RES.append(r)
    log(f"  [{len(RES):3d}/{len(jobs)}] {BENCHMARKS[k]['label']:18s} {lbl:22s} seed {s}  "
        f"{METRIC[k]} {r['score']:.4f}  ({r['seconds']/60:.1f} min)  elapsed {(time.time()-t0)/60:.1f} min")
R = pd.DataFrame(RES)
log(f"all runs done in {(time.time()-t0)/60:.1f} min")

## 4. XGBoost on the same splits, tuned per dataset

In [ ]:
from xgboost import XGBClassifier, XGBRegressor
XGB = {}
for k in KEYS:
    d, a, t0 = DATA[k], DATA[k].arrays(), time.time()
    hib = higher_is_better(k)
    def make(params, seed):
        common = dict(tree_method="hist", device=str(DEVICE), random_state=seed,
                      n_estimators=2000, early_stopping_rounds=50, **params)
        return XGBRegressor(**common) if d.task == "regression" else XGBClassifier(**common)
    def raw(m, X):
        if d.task == "regression": return m.predict(X)
        if d.task == "binary":
            p = np.clip(m.predict_proba(X)[:, 1], 1e-7, 1 - 1e-7); return np.log(p / (1 - p))
        return m.predict_proba(X)
    def objective(trial):
        params = {"max_depth": trial.suggest_int("max_depth", 3, 10),
                  "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.5, log=True),
                  "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                  "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                  "min_child_weight": trial.suggest_float("min_child_weight", 1e-4, 100.0, log=True),
                  "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True)}
        m = make(params, 0).fit(a["Xtr"], a["ytr"], eval_set=[(a["Xva"], a["yva"])], verbose=False)
        return score_from_raw(d, d.y_va, raw(m, a["Xva"]))
    st = optuna.create_study(direction="maximize" if hib else "minimize",
                             sampler=optuna.samplers.TPESampler(seed=0, multivariate=True))
    st.optimize(objective, n_trials=XGB_TRIALS)
    XGB[k] = np.array([score_from_raw(d, d.y_te, raw(make(st.best_params, s).fit(
        a["Xtr"], a["ytr"], eval_set=[(a["Xva"], a["yva"])], verbose=False), a["Xte"])) for s in SEEDS])
    log(f"[xgb] {BENCHMARKS[k]['label']:18s} ours {XGB[k].mean():.4f} | published {published(k,'XGBoost',kind='gbdt')} "
        f"({(time.time()-t0)/60:.1f} min)")

## 5. Results

In [ ]:
g = R.groupby(["label", "key"])["score"]
piv = g.mean().unstack("key")
sdv = g.std(ddof=1).unstack("key").reindex_like(piv).fillna(0.0)
order = [m[0] for m in MODELS]
rows = []
for lbl in order:
    row = {"model": lbl, "params": R[(R.label == lbl) & (R.key == KEYS[0])]["n_params"].iloc[0]
           if len(R[(R.label == lbl) & (R.key == KEYS[0])]) else np.nan}
    for k in KEYS:
        row[BENCHMARKS[k]["label"]] = f"{piv.loc[lbl, k]:.4f} ± {sdv.loc[lbl, k]:.4f}"
    rows.append(row)
rows.append({"model": "XGBoost (ours, tuned)", "params": np.nan,
             **{BENCHMARKS[k]["label"]: f"{XGB[k].mean():.4f} ± {XGB[k].std(ddof=1):.4f}" for k in KEYS}})
for m in ("MLP", "ResNet", "NODE", "FT-Transformer"):
    rows.append({"model": f"{m} (published)", "params": np.nan,
                 **{BENCHMARKS[k]["label"]: published(k, m) for k in KEYS}})
rows.append({"model": "XGBoost (published)", "params": np.nan,
             **{BENCHMARKS[k]["label"]: published(k, "XGBoost", kind="gbdt") for k in KEYS}})
print("Test scores — accuracy for AD/JA/CO (higher better), RMSE for CA/YE (lower better)")
show(pd.DataFrame(rows).set_index("model"), {"params": "{:,.0f}"})

In [ ]:
fig, axes = plt.subplots(1, len(KEYS), figsize=(5.5 * len(KEYS), 5.4))
for ax, k in zip(np.atleast_1d(axes), KEYS):
    hib = higher_is_better(k)
    entries = [(lbl, piv.loc[lbl, k], sdv.loc[lbl, k], "#7b5cff") for lbl in order]
    entries.append(("XGBoost (ours)", XGB[k].mean(), XGB[k].std(ddof=1), "#ff8a3d"))
    for m in ("XGBoost", None):
        pass
    entries.append(("XGBoost (published)", published(k, "XGBoost", kind="gbdt"), np.nan, "#9aa0a6"))
    entries.append(("FT-Transformer (published)", published(k, "FT-Transformer"), np.nan, "#9aa0a6"))
    entries.append(("MLP (published)", published(k, "MLP"), np.nan, "#9aa0a6"))
    entries.sort(key=lambda e: e[1], reverse=not hib)
    for i, (lbl, mu, sd, c) in enumerate(entries):
        ax.errorbar(mu, i, xerr=(0 if (sd is None or np.isnan(sd)) else sd), fmt="o", color=c,
                    ms=8, capsize=4, mec="black" if lbl.startswith("v4") else c)
        ax.text(mu, i + 0.3, f"{mu:.3f}", ha="center", fontsize=8, color=c)
    ax.set_yticks(range(len(entries))); ax.set_yticklabels([e[0] for e in entries], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(f"test {METRIC[k]} ({'higher' if hib else 'lower'} better)")
    ax.set_title(BENCHMARKS[k]["label"], fontweight="bold")
plt.suptitle("purple = v4 · orange = our XGBoost · grey = published reference points", y=1.02, fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Findings

In [ ]:
sep = "=" * 96
print(sep); print("FINDINGS — v4 vs the literature, 5 datasets"); print(sep)
best_v4 = {k: min(((piv.loc[l, k], l) for l in order if l.startswith("v4")),
                  key=lambda z: z[0] if not higher_is_better(k) else -z[0]) for k in KEYS}

print("\n1. v4's best arm vs. published references")
for k in KEYS:
    hib = higher_is_better(k); v4, lbl = best_v4[k]
    sign = 1 if hib else -1
    print(f"   {BENCHMARKS[k]['label']:18s} best v4 {v4:.4f} ({lbl})")
    print(f"        vs our XGBoost       {sign*(v4-XGB[k].mean()):+.4f}   ({XGB[k].mean():.4f})")
    print(f"        vs published XGBoost {sign*(v4-published(k,'XGBoost',kind='gbdt')):+.4f}   "
          f"({published(k,'XGBoost',kind='gbdt')})")
    print(f"        vs published FT-Transf {sign*(v4-published(k,'FT-Transformer')):+.4f}   "
          f"({published(k,'FT-Transformer')})")
    print(f"        vs published MLP     {sign*(v4-published(k,'MLP')):+.4f}   ({published(k,'MLP')})")

print("\n2. Did per-feature embeddings help on these 5 too?")
for k in KEYS:
    for agg in ("mean-pool", "causal attn"):
        a, b = piv.loc[f"v4 {agg}", k], piv.loc[f"v4 {agg} + PLR", k]
        d = (b - a) if higher_is_better(k) else (a - b)
        print(f"   {BENCHMARKS[k]['label']:18s} {agg:12s} {a:.4f} -> {b:.4f}  ({d:+.4f} in favour of PLR)")

print("\n3. Sanity: does our XGBoost reproduce the published one?")
for k in KEYS:
    pub = published(k, "XGBoost", kind="gbdt")
    d = XGB[k].mean() - pub
    print(f"   {BENCHMARKS[k]['label']:18s} ours {XGB[k].mean():.4f} vs published {pub} ({d:+.4f})")
print("\n" + sep)

## 7. Notes

* v4 was not tuned per dataset; every published number was. The gap that remains after
  tuning v4 (a natural next step, using `tuning.py`'s Optuna machinery) is the honest
  architectural gap.
* No std is reported in this paper's tables, so "published" rows above are single reference
  points, not distributions — read the comparison as "in the right neighbourhood or not",
  not as a statistical test.